# MAD-CORE dataset summary

A tour of the MAD-CORE extxyz shards: what is in them, how the labels are
distributed, whether anything looks broken, and an interactive
[chemiscope](https://chemiscope.org) map of a representative sample.

The notebook sticks to the metatensor-ecosystem idioms:

* **metatomic** `System`s (via `systems_to_torch`) with `vesin.metatomic`
  neighbour lists for geometry checks, in float64 and explicit units;
* **metatensor** `TensorMap`s for the targets (energy, with forces / virial
  stored as *gradients*, as metatrain expects them) and for descriptors
  (featomic SOAP reduced with `keys_to_properties` / `mean_over_samples`);
* **chemiscope** input built from explicit `properties` (with units and
  descriptions), force arrows as `shapes`, map `settings`, saved with
  `write_input` so the `.json.gz` can be opened on chemiscope.org, plus the
  PET-MAD latent map through `chemiscope.metatomic_featurizer`.

Get the data first (the record is a draft, so it needs a preview token):

```bash
MC_TOKEN=<preview-token> bash ~/metawork/etc/download-madcore-extxyz.sh
```

and point `MADCORE_DIR` at it if it is not in `~/data/madcore`. The
notebook is written against whatever keys the files actually carry
(energy / forces / stress names and the subset label are discovered, not
hard-coded), so it also runs on other extxyz datasets.

In [ ]:
import collections
import glob
import itertools
import os
import pickle
import warnings
from pathlib import Path

import ase.io
import chemiscope
import matplotlib.pyplot as plt
import metatensor.torch as mts
import metatomic.torch as mta
import numpy as np
import pandas as pd
import torch
from ase.data import chemical_symbols
from ase.stress import voigt_6_to_full_3x3_stress
from IPython.display import display

try:
    from metatomic_ase import MetatomicCalculator
except ImportError:  # older metatomic ships it inside metatomic.torch
    from metatomic.torch.ase_calculator import MetatomicCalculator

warnings.filterwarnings("ignore", message=".*vesin.metatomic was only tested.*")
import vesin.metatomic  # noqa: E402

DATA_DIR = Path(os.environ.get("MADCORE_DIR", "~/data/madcore")).expanduser()
OUT_DIR = Path(os.environ.get("MADCORE_OUT", "madcore-outputs"))
N_SAMPLE = int(os.environ.get("MADCORE_N_SAMPLE", 1000))  # structures kept in memory
SEED = 0
PET_MAD = Path(
    os.environ.get(
        "PET_MAD_MODEL",
        Path("~/metawork/openmm-metatomic/models/pet-mad-xs-v1.5.0.pt").expanduser(),
    )
)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUT_DIR.mkdir(exist_ok=True)
shards = sorted(glob.glob(str(DATA_DIR / "*.extxyz")) + glob.glob(str(DATA_DIR / "*.xyz")))
assert shards, f"no extxyz shards in {DATA_DIR} -- run etc/download-madcore-extxyz.sh"
print(f"{len(shards)} shards in {DATA_DIR}, device={DEVICE}")

## 1. Inventory

Counting frames only needs the atom-count header lines, so this is much
faster than parsing the files.

In [ ]:
def count_frames(path):
    n = 0
    with open(path) as f:
        for line in f:
            if not line.strip():
                continue
            n += 1
            for _ in itertools.islice(f, int(line) + 1):
                pass
    return n


inventory = pd.DataFrame(
    [
        {"shard": Path(p).name, "size_MB": os.path.getsize(p) / 2**20, "n_frames": count_frames(p)}
        for p in shards
    ]
)
display(inventory.style.format({"size_MB": "{:.1f}"}))
n_total = inventory.n_frames.sum()
print(f"{n_total:,} structures, {inventory.size_MB.sum() / 1024:.2f} GB in total")

## 2. One streaming pass

Every structure is reduced to a row of cheap scalars (so the statistics
below cover the *whole* dataset), and a uniform random subset of
`N_SAMPLE` structures is kept as `ase.Atoms` for the geometry, descriptor
and chemiscope sections. The result is cached next to the data.

In [ ]:
rng = np.random.default_rng(SEED)
keep = set(rng.choice(n_total, size=min(N_SAMPLE, n_total), replace=False).tolist())
cache = DATA_DIR / f".summary-cache-{n_total}-{N_SAMPLE}-{SEED}.pkl"


def full_stress(s):
    s = np.asarray(s, float).reshape(-1)
    return s.reshape(3, 3) if s.size == 9 else voigt_6_to_full_3x3_stress(s)


def pick(keys, *candidates):
    return next((c for c in candidates if c in keys), None)


def scan():
    rows, sample = [], []
    info_keys, array_keys = collections.Counter(), collections.Counter()
    gid = 0
    for shard in shards:
        for local, atoms in enumerate(ase.io.iread(shard, index=":", format="extxyz")):
            info = dict(atoms.info)
            if atoms.calc is not None:  # ase moves energy/forces/stress onto a calculator
                info.update({k: v for k, v in atoms.calc.results.items() if k != "forces"})
            forces = atoms.arrays.get("forces")
            if forces is None and atoms.calc is not None:
                forces = atoms.calc.results.get("forces")
            info_keys.update(info.keys())
            array_keys.update([*atoms.arrays, *(["forces"] if forces is not None and "forces" not in atoms.arrays else [])])
            rows.append(
                {
                    "id": gid,
                    "shard": Path(shard).name,
                    "index": local,
                    "formula": atoms.get_chemical_formula(mode="hill"),
                    "n_atoms": len(atoms),
                    "numbers": np.bincount(atoms.numbers, minlength=119),
                    "pbc": "".join("TF"[not p] for p in atoms.pbc),
                    "volume": atoms.get_volume() if atoms.pbc.all() else np.nan,
                    "max_force": np.linalg.norm(forces, axis=1).max() if forces is not None else np.nan,
                    "rms_force": np.sqrt((forces**2).sum(1).mean()) if forces is not None else np.nan,
                    **{k: v for k, v in info.items() if np.isscalar(v) or np.size(v) in (6, 9)},
                }
            )
            if gid in keep:
                if forces is not None:
                    atoms.arrays["forces"] = np.asarray(forces)
                atoms.info.update(info)
                atoms.calc = None
                atoms.info["id"] = gid
                sample.append(atoms)
            gid += 1
    return pd.DataFrame(rows), sample, info_keys, array_keys


if cache.exists():
    df, sample, info_keys, array_keys = pickle.loads(cache.read_bytes())
else:
    df, sample, info_keys, array_keys = scan()
    cache.write_bytes(pickle.dumps((df, sample, info_keys, array_keys)))
print(f"summarised {len(df):,} structures, kept {len(sample)} in memory")

### Schema

Which keys the files carry, and on how many structures. The energy,
stress and subset columns used below are picked from these.

In [ ]:
schema = pd.DataFrame(
    [("info", k, v) for k, v in info_keys.items()] + [("arrays", k, v) for k, v in array_keys.items()],
    columns=["where", "key", "n_structures"],
).assign(coverage=lambda d: d.n_structures / len(df))
display(schema.sort_values(["where", "n_structures"], ascending=[True, False]).style.format({"coverage": "{:.1%}"}))

E_KEY = pick(info_keys, "energy", "free_energy", "total_energy", "dft_energy", "E")
S_KEY = pick(info_keys, "stress", "dft_stress", "REF_stress")
SUBSET_KEY = pick(info_keys, "subset", "mad_subset", "dataset", "config_type", "source", "origin", "label")
if SUBSET_KEY is None:
    df["subset"], SUBSET_KEY = df.shard, "subset"
df[SUBSET_KEY] = df[SUBSET_KEY].fillna("<none>").astype(str)
print(f"energy: {E_KEY!r}  stress: {S_KEY!r}  subset: {SUBSET_KEY!r}")

## 3. Composition and size

In [ ]:
counts = np.stack(df.numbers.to_numpy())  # (n_structures, 119) atoms of each Z
present = np.flatnonzero(counts.sum(0))
elements = [chemical_symbols[z] for z in present]
df["n_species"] = (counts > 0).sum(1)

subsets = df.groupby(SUBSET_KEY).agg(
    structures=("id", "size"),
    atoms=("n_atoms", "sum"),
    median_atoms=("n_atoms", "median"),
    max_atoms=("n_atoms", "max"),
    species=("n_species", "median"),
    periodic=("pbc", lambda p: (p == "TTT").mean()),
)
display(subsets.sort_values("structures", ascending=False).style.format({"periodic": "{:.0%}", "species": "{:.0f}"}))
print(
    f"{len(df):,} structures, {df.n_atoms.sum():,} atoms, {len(elements)} elements:",
    " ".join(elements),
)
print("pbc patterns:", df.pbc.value_counts().to_dict())

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
z = np.arange(present.min(), present.max() + 1)
ax[0].bar(z, (counts[:, z] > 0).sum(0), color="C0")
ax[0].set(ylabel="structures containing", yscale="log")
ax[1].bar(z, counts[:, z].sum(0), color="C1")
ax[1].set(ylabel="atoms", yscale="log")
ax[1].set_xticks(z, [chemical_symbols[i] for i in z], rotation=90, fontsize=7)
fig.suptitle("Element coverage")
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
for name, g in df.groupby(SUBSET_KEY):
    ax[0].hist(g.n_atoms, bins=np.logspace(0, np.log10(df.n_atoms.max() + 1), 40), histtype="step", label=name)
ax[0].set(xscale="log", xlabel="atoms per structure", ylabel="structures")
ax[1].hist(df.n_species, bins=np.arange(0.5, df.n_species.max() + 1.5), color="C2")
ax[1].set(xlabel="species per structure")
density = df.n_atoms / df.volume
ax[2].hist(density.dropna(), bins=50, color="C3")
ax[2].set(xlabel="number density (atoms/Å³, periodic only)")
ax[0].legend(fontsize=7)
fig.tight_layout()

## 4. Labels

Raw energies are dominated by the composition, so besides energy per atom
we fit a per-element linear baseline $E \approx \sum_Z n_Z\,\mu_Z$ (the
same composition model metatrain uses before training) and look at the
residual. The residual is what a model actually has to learn, and large
values point to suspicious labels.

In [ ]:
has_E = df[E_KEY].notna() if E_KEY else pd.Series(False, index=df.index)
if E_KEY:
    E = df.loc[has_E, E_KEY].to_numpy(float)
    X = counts[has_E.to_numpy()][:, present].astype(float)
    mu, *_ = np.linalg.lstsq(X, E, rcond=None)
    df.loc[has_E, "energy_per_atom"] = E / df.loc[has_E, "n_atoms"]
    df.loc[has_E, "residual_per_atom"] = (E - X @ mu) / df.loc[has_E, "n_atoms"]
    display(
        pd.DataFrame({"element": elements, "mu_eV": mu, "atoms": X.sum(0).astype(int)})
        .set_index("element")
        .T.style.format("{:.3f}", subset=pd.IndexSlice["mu_eV", :])
    )
    print("composition-baseline residual per atom: MAE", np.abs(df.residual_per_atom).mean(), "eV/atom")

if S_KEY:
    stress = df.loc[df[S_KEY].notna(), S_KEY]
    df.loc[stress.index, "pressure_GPa"] = stress.map(lambda s: -np.trace(full_stress(s)) / 3 * 160.21766)  # eV/Å³ -> GPa

label_stats = df.groupby(SUBSET_KEY)[
    [c for c in ["energy_per_atom", "residual_per_atom", "max_force", "rms_force", "pressure_GPa"] if c in df]
].describe(percentiles=[0.5, 0.99]).T
display(label_stats.style.format("{:.3g}"))

In [ ]:
cols = [c for c in ["energy_per_atom", "residual_per_atom", "max_force", "pressure_GPa"] if c in df and df[c].notna().any()]
fig, ax = plt.subplots(1, len(cols), figsize=(4.2 * len(cols), 3.5))
ax = np.atleast_1d(ax)
units = {"energy_per_atom": "eV/atom", "residual_per_atom": "eV/atom", "max_force": "eV/Å", "pressure_GPa": "GPa"}
for a, c in zip(ax, cols):
    v = df[c].dropna()
    lo, hi = np.percentile(v, [0.1, 99.9])
    bins = np.logspace(np.log10(max(v[v > 0].min(), 1e-4)), np.log10(v.max()), 50) if c == "max_force" else np.linspace(lo, hi, 50)
    for name, g in df.groupby(SUBSET_KEY):
        a.hist(g[c].dropna(), bins=bins, histtype="step", label=name)
    a.set(xlabel=f"{c} ({units[c]})", yscale="log", xscale="log" if c == "max_force" else "linear")
ax[0].legend(fontsize=7)
fig.tight_layout()

### Outliers

Structures more than 5 robust standard deviations away from their
subset's median residual, or with very large forces, are worth a look
before training on them.

In [ ]:
flags = pd.Series(False, index=df.index)
if "residual_per_atom" in df:
    g = df.groupby(SUBSET_KEY).residual_per_atom
    mad = g.transform(lambda r: 1.4826 * (r - r.median()).abs().median())
    flags |= (df.residual_per_atom - g.transform("median")).abs() > 5 * mad
flags |= df.max_force > 50.0
df["outlier"] = flags
print(f"{flags.sum()} flagged structures ({flags.mean():.2%})")
display(df.loc[flags, ["id", "shard", "index", SUBSET_KEY, "formula", "n_atoms", "residual_per_atom", "max_force"]].head(20))

## 5. metatomic systems and neighbour lists

The sample is converted to metatomic `System`s (float64, Å) and given a
half neighbour list from `vesin.metatomic`, exactly like an engine would
prepare them for a model. The shortest pair distance per structure is a
quick check for overlapping atoms.

In [ ]:
systems = mta.systems_to_torch(sample, dtype=torch.float64)
NL_OPTIONS = mta.NeighborListOptions(cutoff=6.0, full_list=False, strict=True)
nl_calc = vesin.metatomic.NeighborList(NL_OPTIONS, length_unit="angstrom")
for system in systems:
    system.add_neighbor_list(NL_OPTIONS, nl_calc.compute(system))

print(systems[0])
nl = systems[0].get_neighbor_list(NL_OPTIONS)
print(nl)


def min_distance(system):
    d = system.get_neighbor_list(NL_OPTIONS).values.reshape(-1, 3).norm(dim=1)
    return d.min().item() if len(d) else np.nan


sample_df = df.set_index("id").loc[[a.info["id"] for a in sample]].reset_index()
sample_df["min_distance"] = [min_distance(s) for s in systems]
sample_df["neighbors_per_atom"] = [2 * len(s.get_neighbor_list(NL_OPTIONS).samples) / len(s) for s in systems]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(sample_df.min_distance.dropna(), bins=60)
ax[0].set(xlabel="shortest interatomic distance (Å)", ylabel="structures")
ax[1].hist(sample_df.neighbors_per_atom, bins=60, color="C1")
ax[1].set(xlabel=f"neighbours within {NL_OPTIONS.cutoff} Å per atom")
fig.tight_layout()
display(sample_df.nsmallest(5, "min_distance")[["id", SUBSET_KEY, "formula", "min_distance"]])

## 6. Targets as metatensor `TensorMap`s

metatrain stores an energy target as a single block with one sample per
system; forces live in the `positions` gradient (with the sign flipped,
$\partial E/\partial r = -F$) and the virial in the `strain` gradient
($\partial E/\partial\varepsilon = -\sigma V$). Building that layout here
checks that every sampled structure has consistent labels, and the saved
`.mts` file can be reloaded with `metatensor.torch.load`.

In [ ]:
def energy_tensormap(frames):
    E = torch.tensor([a.info[E_KEY] for a in frames], dtype=torch.float64).reshape(-1, 1)
    block = mts.TensorBlock(
        values=E,
        samples=mts.Labels("system", torch.arange(len(frames)).reshape(-1, 1)),
        components=[],
        properties=mts.Labels("energy", torch.tensor([[0]])),
    )
    xyz = mts.Labels("xyz", torch.arange(3).reshape(-1, 1))
    grad_samples = torch.tensor([(i, i, j) for i, a in enumerate(frames) for j in range(len(a))])
    forces = torch.tensor(np.concatenate([a.arrays["forces"] for a in frames]), dtype=torch.float64)
    block.add_gradient(
        "positions",
        mts.TensorBlock(
            values=-forces.reshape(-1, 3, 1),
            samples=mts.Labels(["sample", "system", "atom"], grad_samples),
            components=[xyz],
            properties=block.properties,
        ),
    )
    periodic = [i for i, a in enumerate(frames) if S_KEY and S_KEY in a.info and a.pbc.all()]
    if periodic:
        virial = torch.tensor(
            np.stack([-full_stress(frames[i].info[S_KEY]) * frames[i].get_volume() for i in periodic]),
            dtype=torch.float64,
        )
        xyz_1, xyz_2 = (mts.Labels(n, torch.arange(3).reshape(-1, 1)) for n in ("xyz_1", "xyz_2"))
        block.add_gradient(
            "strain",
            mts.TensorBlock(
                values=virial.reshape(-1, 3, 3, 1),
                samples=mts.Labels(["sample"], torch.tensor(periodic).reshape(-1, 1)),
                components=[xyz_1, xyz_2],
                properties=block.properties,
            ),
        )
    return mts.TensorMap(mts.Labels.single(), [block])


labelled = [a for a in sample if E_KEY in a.info and "forces" in a.arrays]
energy = energy_tensormap(labelled)
print(energy)
print(energy.block())
mts.save(OUT_DIR / "madcore-sample-energy.mts", energy)
assert mts.equal(mts.load(OUT_DIR / "madcore-sample-energy.mts"), energy)

## 7. SOAP descriptors with featomic + metatensor

MAD-CORE spans most of the periodic table, so a standard SOAP (one channel
per pair of neighbour species) would be enormous. Instead we compute an
*element-agnostic* SOAP (every atom relabelled as the same species): it
describes geometry only, which is complementary to the chemically-aware
PET-MAD map in the next section. The atomic TensorMap is reduced to one
vector per structure with metatensor operations, then projected with PCA.

In [ ]:
import featomic.torch  # noqa: E402

HYPERS = {
    "cutoff": {"radius": 5.0, "smoothing": {"type": "ShiftedCosine", "width": 0.5}},
    "density": {"type": "Gaussian", "width": 0.4},
    "basis": {"type": "TensorProduct", "max_angular": 4, "radial": {"type": "Gto", "max_radial": 6}},
}
soap_calc = featomic.torch.SoapPowerSpectrum(**HYPERS)


def agnostic(system):
    return mta.System(torch.ones_like(system.types), system.positions, system.cell, system.pbc)


agnostic_systems = [agnostic(s) for s in systems]
for options in soap_calc.requested_neighbor_lists():
    calc = vesin.metatomic.NeighborList(options, length_unit="angstrom")
    for s in agnostic_systems:
        s.add_neighbor_list(options, calc.compute(s))
soap = soap_calc.compute(agnostic_systems)
soap = soap.keys_to_properties(["neighbor_1_type", "neighbor_2_type"]).keys_to_samples("center_type")
print(soap)
soap_struct = mts.mean_over_samples(soap, sample_names=["atom", "center_type"])
feats = soap_struct.block().values.numpy()
feats = feats / np.linalg.norm(feats, axis=1, keepdims=True)


def pca(x, k=3):
    x = x - x.mean(0)
    u, s, _ = np.linalg.svd(x, full_matrices=False)
    print(f"PCA explained variance: {np.round(s[:k] ** 2 / (s**2).sum(), 3)}")
    return u[:, :k] * s[:k]


def fps(x, n):
    idx = [0]
    d = np.linalg.norm(x - x[0], axis=1)
    for _ in range(n - 1):
        idx.append(int(d.argmax()))
        d = np.minimum(d, np.linalg.norm(x - x[idx[-1]], axis=1))
    return np.array(idx)


soap_pca = pca(feats)
fps_rank = np.full(len(sample), len(sample))
fps_rank[fps(feats, min(100, len(sample)))] = np.arange(min(100, len(sample)))

fig, ax = plt.subplots(figsize=(6, 5))
for name in sorted(sample_df[SUBSET_KEY].unique()):  # same order (and colours) as groupby
    m = (sample_df[SUBSET_KEY] == name).to_numpy()
    ax.scatter(soap_pca[m, 0], soap_pca[m, 1], s=6, label=name)
ax.set(xlabel="SOAP PC1", ylabel="SOAP PC2", title="element-agnostic SOAP")
ax.legend(fontsize=7, markerscale=2)

## 8. PET-MAD latent space (metatomic featurizer)

`chemiscope.metatomic_featurizer` runs any metatomic model with a
`features` output; PET-MAD's last-layer features give a map that knows
about chemistry. The same model's energy is compared to the dataset's,
after the same composition baseline, as a sanity check of the labels
(differences in the reference level of theory will show up here too).

In [ ]:
pet = None
if PET_MAD.exists():
    model = mta.load_atomistic_model(str(PET_MAD)).to(DEVICE)  # the featurizer moves systems, not the model
    featurize = chemiscope.metatomic_featurizer(model, device=DEVICE)
    batches = lambda n: (sample[i : i + n] for i in range(0, len(sample), n))  # noqa: E731
    pet_feats = np.concatenate([featurize(b, None) for b in batches(16)])
    pet_pca = pca(pet_feats / np.linalg.norm(pet_feats, axis=1, keepdims=True))

    calc = MetatomicCalculator(model, device=DEVICE)
    pet_E = np.concatenate([np.atleast_1d(calc.compute_energy(b)["energy"]) for b in batches(16)])
    comp = np.stack([np.bincount(a.numbers, minlength=119)[present] for a in sample]).astype(float)
    mu_pet, *_ = np.linalg.lstsq(comp, pet_E, rcond=None)
    sample_df["pet_residual_per_atom"] = (pet_E - comp @ mu_pet) / sample_df.n_atoms
    pet = True

    fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
    for name in sorted(sample_df[SUBSET_KEY].unique()):  # same order (and colours) as groupby
        m = (sample_df[SUBSET_KEY] == name).to_numpy()
        ax[0].scatter(pet_pca[m, 0], pet_pca[m, 1], s=6, label=name)
        ax[1].scatter(sample_df.residual_per_atom[m], sample_df.pet_residual_per_atom[m], s=6)
    ax[0].set(xlabel="PET-MAD PC1", ylabel="PET-MAD PC2")
    ax[0].legend(fontsize=7, markerscale=2)
    lim = np.nanpercentile(np.abs(sample_df[["residual_per_atom", "pet_residual_per_atom"]]), 99.5)
    ax[1].plot([-lim, lim], [-lim, lim], "k--", lw=0.8)
    ax[1].set(xlabel="dataset residual (eV/atom)", ylabel="PET-MAD residual (eV/atom)", xlim=(-lim, lim), ylim=(-lim, lim))
    fig.tight_layout()
    print("PET-MAD vs dataset residual MAE:", np.nanmean(np.abs(sample_df.pet_residual_per_atom - sample_df.residual_per_atom)), "eV/atom")
else:
    print(f"no PET-MAD model at {PET_MAD}; set PET_MAD_MODEL to enable this section")

## 9. Interactive chemiscope map

Properties are passed explicitly (with units and descriptions) rather
than dumped from `atoms.info`, forces are shown as arrows, and the input
is written to `madcore-outputs/madcore-sample.json.gz`, which can be
opened on <https://chemiscope.org> or with `chemiscope.show_input`
without re-running anything.

In [ ]:
def prop(values, description, units=None):
    return {"target": "structure", "values": np.asarray(values).tolist(), "description": description, **({"units": units} if units else {})}


properties = {
    "subset": prop(sample_df[SUBSET_KEY].astype(str), "dataset subset"),
    "formula": prop(sample_df.formula, "chemical formula"),
    "shard": prop(sample_df.shard, "source file"),
    "n_atoms": prop(sample_df.n_atoms, "number of atoms"),
    "n_species": prop(sample_df.n_species, "number of elements"),
    "min distance": prop(sample_df.min_distance.fillna(-1), "shortest interatomic distance", "Å"),
    "max |F|": prop(sample_df.max_force.fillna(0), "largest atomic force", "eV/Å"),
    "SOAP PC1": prop(soap_pca[:, 0], "element-agnostic SOAP, PC1"),
    "SOAP PC2": prop(soap_pca[:, 1], "element-agnostic SOAP, PC2"),
    "FPS rank": prop(fps_rank, "farthest-point-sampling rank in SOAP space (lower = more diverse)"),
    "outlier": prop(sample_df.outlier.map({True: "flagged", False: "ok"}), "5σ residual / large force flag"),
}
for col, desc, unit in [
    ("energy_per_atom", "energy per atom", "eV"),
    ("residual_per_atom", "energy per atom minus composition baseline", "eV"),
    ("pressure_GPa", "pressure from the stress", "GPa"),
    ("pet_residual_per_atom", "PET-MAD energy per atom minus composition baseline", "eV"),
]:
    if col in sample_df and sample_df[col].notna().all():
        properties[col.replace("_", " ")] = prop(sample_df[col], desc, unit)
if pet:
    properties.update({f"PET-MAD PC{i + 1}": prop(pet_pca[:, i], f"PET-MAD features, PC{i + 1}") for i in range(3)})

shapes = {}
if all("forces" in a.arrays for a in sample):
    shapes["forces"] = chemiscope.ase_vectors_to_arrows(sample, "forces", scale=0.5)

x, y = ("PET-MAD PC1", "PET-MAD PC2") if pet else ("SOAP PC1", "SOAP PC2")
settings = chemiscope.quick_settings(
    x=x, y=y, map_color="residual per atom" if "residual per atom" in properties else "n_atoms", symbol="subset",
    structure_settings={"unitCell": True, **({"shape": "forces"} if shapes else {})},
)
metadata = {
    "name": "MAD-CORE: random sample",
    "description": f"{len(sample)} of {len(df):,} structures sampled uniformly from {DATA_DIR.name}",
    "authors": ["metawork notebooks/madcore-dataset-summary"],
}
chemiscope.write_input(
    str(OUT_DIR / "madcore-sample.json.gz"), structures=sample, properties=properties, shapes=shapes, settings=settings, metadata=metadata
)
chemiscope.show(sample, properties=properties, shapes=shapes, settings=settings, metadata=metadata)

## 10. Summary

In [ ]:
summary = {
    "structures": f"{len(df):,}",
    "atoms": f"{df.n_atoms.sum():,}",
    "elements": len(elements),
    "subsets": df[SUBSET_KEY].nunique(),
    "periodic (TTT)": f"{(df.pbc == 'TTT').mean():.1%}",
    "atoms/structure (median, max)": f"{df.n_atoms.median():.0f}, {df.n_atoms.max()}",
    "energy key / coverage": f"{E_KEY} / {has_E.mean():.1%}",
    "stress key / coverage": f"{S_KEY} / {df[S_KEY].notna().mean():.1%}" if S_KEY else "none",
    "composition-baseline MAE (eV/atom)": f"{np.abs(df.residual_per_atom).mean():.3f}" if "residual_per_atom" in df else "n/a",
    "flagged outliers": f"{df.outlier.sum()} ({df.outlier.mean():.2%})",
    "shortest distance in sample (Å)": f"{sample_df.min_distance.min():.2f}",
}
display(pd.Series(summary, name="MAD-CORE").to_frame())